2. **Construct SPARQL & Return List of Labels, IRIs, and Axioms**  
    - Study the `week5_endpoint_client.ipynb` to understand how to leverage rdflib with SPARQL
    - Construct SPARQL queries for each ontology that are designed to return a complete list of classes and associated axioms. Note: Do not return deprecated classes; your query should a file with a column for IRIs, a column for rdfs:labels, and a column for axioms associated with each IRI 
    - Successfully run your SPARQL queries in a Jupyter notebook, named `class-axiom-generator.ipynb`; save this in the `notebooks` directory
    - Ensure your code saves the resulting lists following the convention: `ontology abbreviation-axioms.xlsx`. Note: Be sure your xlsx file has a column for "iri" and a column for "axiom"

In [1]:
from pathlib import Path
import pandas as pd
from rdflib import Graph, URIRef
from rdflib.namespace import OWL, RDFS, XSD

In [6]:
NB_DIR = Path.cwd().resolve()
SRC_DIR = None
for p in [NB_DIR] + list(NB_DIR.parents):
    candidate = p / "assignment" / "src"
    if candidate.exists():
        SRC_DIR = candidate
        break
    candidate = p / "src"  # also allow projects that don't have the extra 'assignment' level
    if candidate.exists():
        SRC_DIR = candidate
        break

if SRC_DIR is None:
    raise FileNotFoundError(
        f"Could not find a 'src' directory above {NB_DIR}. "
        "If your project root is at e.g. .../assignment/, open the notebook there or set SRC_DIR manually."
    )

DATA_DIR = SRC_DIR / "data"

Define TTL and OUT

In [7]:
TTL = {
    "bfo":  SRC_DIR / "bfo-core.ttl",
    "ies":  SRC_DIR / "ies.ttl",
    "qudt": SRC_DIR / "qudt.ttl",
    "ccom": SRC_DIR / "ccom.ttl",
    "ccot": SRC_DIR / "ccot.ttl",
    "to":   SRC_DIR / "time.ttl",
}

OUT = {
    "bfo":  DATA_DIR / "bfo-class-axioms.xlsx",
    "ies":  DATA_DIR / "ies-class-axioms.xlsx",
    "qudt": DATA_DIR / "qudt-class-axioms.xlsx",
    "ccom": DATA_DIR / "ccom-class-axioms.xlsx",
    "ccot": DATA_DIR / "ccot-class-axioms.xlsx",
    "to":   DATA_DIR / "to-class-axioms.xlsx",
}

In [9]:
CLASS_AXIOM_QUERY = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:  <http://www.w3.org/2002/07/owl#>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
SELECT ?iri ?label ?axiom
WHERE {
  ?iri a ?classType .
  VALUES ?classType { owl:Class rdfs:Class }
  OPTIONAL { ?iri rdfs:label ?label }

  ?iri ?predicate ?axiom .

  FILTER NOT EXISTS {
    ?iri owl:deprecated ?d .
    FILTER( ?d = true || lcase(str(?d)) = "true" )
  }

  FILTER ( !isBlank(?iri) )   
}
ORDER BY ?label
"""

In [10]:
def run_local_select(ttl_path: Path, query: str) -> pd.DataFrame:
    g = Graph()
    g.parse(ttl_path)  # rdflib guesses Turtle from .ttl
    res = g.query(query)
    rows = []
    for row in res:
        d = row.asdict()  # {'iri': rdflib.term.URIRef(...), 'label': rdflib.term.Literal(...)} or missing
        iri_term   = d.get("iri")
        label_term = d.get("label")
        axiom_term = d.get("axiom")
        rows.append({
            "iri":   str(iri_term)   if iri_term is not None else None,
            "label": str(label_term) if label_term is not None else None,
            "axiom": str(axiom_term) if axiom_term is not None else None,
        })
    return pd.DataFrame(rows, columns=["iri", "label", "axiom"])

In [11]:
for key, ttl_path in TTL.items():
    if not ttl_path.exists():
        print(f"⚠️  Missing {ttl_path} — skipping {key}")
        continue
    print(f"Querying {ttl_path} …")
    df = run_local_select(ttl_path, CLASS_AXIOM_QUERY)
    df.to_excel(OUT[key], index=False)
    print(f"✅ Saved {len(df)} rows to {OUT[key]}")

Querying /Users/dc2666/GitHub/Ontology-Tradecraft/projects/project-3/assignment/src/bfo-core.ttl …
✅ Saved 261 rows to /Users/dc2666/GitHub/Ontology-Tradecraft/projects/project-3/assignment/src/data/bfo-class-axioms.xlsx
Querying /Users/dc2666/GitHub/Ontology-Tradecraft/projects/project-3/assignment/src/ies.ttl …
✅ Saved 2629 rows to /Users/dc2666/GitHub/Ontology-Tradecraft/projects/project-3/assignment/src/data/ies-class-axioms.xlsx
Querying /Users/dc2666/GitHub/Ontology-Tradecraft/projects/project-3/assignment/src/qudt.ttl …
✅ Saved 692 rows to /Users/dc2666/GitHub/Ontology-Tradecraft/projects/project-3/assignment/src/data/qudt-class-axioms.xlsx
Querying /Users/dc2666/GitHub/Ontology-Tradecraft/projects/project-3/assignment/src/ccom.ttl …
✅ Saved 168 rows to /Users/dc2666/GitHub/Ontology-Tradecraft/projects/project-3/assignment/src/data/ccom-class-axioms.xlsx
Querying /Users/dc2666/GitHub/Ontology-Tradecraft/projects/project-3/assignment/src/ccot.ttl …
✅ Saved 216 rows to /Users/dc26